# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [2]:
from pathlib import Path
import pandas as pd

dataset_path = Path('../first/delivery_dataset.csv')
if not dataset_path.exists():
    dataset_path = Path('first/delivery_dataset.csv')

df = pd.read_csv(dataset_path)

print('Размер датасета:', df.shape)
display(df.head())


Размер датасета: (500, 8)


,Дата заказа (ГГГГ-ММ-ДД),Шифр заказа (ID),Расстояние до клиента (в км),Количество позиций в чеке (в шт.),Балл пробок на дорогах (в баллах),Погода (в градусах Цельсия),Этаж доставки (в этажах),Время доставки (в минутах)
0,2026-09-08,ORD-202608001,11.3,1,9,21.0,6,79.2
1,2026-08-29,ORD-202608002,7.3,3,7,27.0,7,46.6
2,2026-08-15,ORD-202608003,2.2,1,3,20.0,5,16.3
3,2026-08-08,ORD-202608004,0.9,2,6,22.9,1,15.8
4,2026-08-21,ORD-202608005,12.3,14,4,18.1,2,69.2


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [3]:
target_col = 'Время доставки (в минутах)'
date_col = 'Дата заказа (ГГГГ-ММ-ДД)'

# ID заказа не несёт полезной информации, а дату кодируем через месяц и день недели.
dates = pd.to_datetime(df[date_col])
X = df.drop(columns=[target_col, date_col, 'Шифр заказа (ID)']).copy()
X['Месяц заказа'] = dates.dt.month.astype(str)
X['День недели заказа'] = dates.dt.dayofweek.astype(str)
X = pd.get_dummies(
    X,
    columns=['Месяц заказа', 'День недели заказа'],
    dtype=float
)
y = df[target_col]

print('Размер матрицы признаков:', X.shape)
display(X.head())


Размер матрицы признаков: (500, 14)


,Расстояние до клиента (в км),Количество позиций в чеке (в шт.),Балл пробок на дорогах (в баллах),Погода (в градусах Цельсия),Этаж доставки (в этажах),Месяц заказа_8,Месяц заказа_9,День недели заказа_0,День недели заказа_1,День недели заказа_2,День недели заказа_3,День недели заказа_4,День недели заказа_5,День недели заказа_6
0,11.3,1,9,21.0,6,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,7.3,3,7,27.0,7,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,2.2,1,3,20.0,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.9,2,6,22.9,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,12.3,14,4,18.1,2,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


3. Разбейте датасет на train val test в отношении 8:1:1

In [4]:
import warnings
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=RuntimeWarning)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_val_scaled = x_scaler.transform(X_val)
X_test_scaled = x_scaler.transform(X_test)
y_train_scaled = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
).ravel()

def add_intercept(X):
    return np.column_stack([np.ones(len(X)), X])

X_train_b = add_intercept(X_train_scaled)
X_val_b = add_intercept(X_val_scaled)
X_test_b = add_intercept(X_test_scaled)

print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)

def predict_original(X, weights):
    prediction_scaled = X @ weights
    return y_scaler.inverse_transform(
        prediction_scaled.reshape(-1, 1)
    ).ravel()

def calculate_metrics(X, y_true, weights):
    prediction = predict_original(X, weights)
    loss = mean_squared_error(y_true, prediction)
    r2 = r2_score(y_true, prediction)
    return loss, r2

def fit_linear_model(
    method, eta, decay=0.0, max_iter=1000,
    tol=1e-8, patience=10, seed=42
):
    rng = np.random.default_rng(seed)
    n_samples, n_features = X_train_b.shape
    weights = np.zeros(n_features)
    velocity = np.zeros(n_features)
    first_moment = np.zeros(n_features)
    second_moment = np.zeros(n_features)
    gradient_memory = np.zeros((n_samples, n_features))
    average_gradient = np.zeros(n_features)
    previous_loss = np.inf
    stable_iterations = 0

    for iteration in range(1, max_iter + 1):
        current_eta = eta / (1 + decay * (iteration - 1))

        if method == 'VGD':
            error = X_train_b @ weights - y_train_scaled
            gradient = 2 * X_train_b.T @ error / n_samples
            weights -= current_eta * gradient

        elif method == 'SGD':
            for index in rng.permutation(n_samples):
                error = X_train_b[index] @ weights - y_train_scaled[index]
                gradient = 2 * error * X_train_b[index]
                weights -= current_eta * gradient

        elif method == 'SAG':
            for index in rng.permutation(n_samples):
                error = X_train_b[index] @ weights - y_train_scaled[index]
                new_gradient = 2 * error * X_train_b[index]
                average_gradient += (
                    new_gradient - gradient_memory[index]
                ) / n_samples
                gradient_memory[index] = new_gradient
                weights -= current_eta * average_gradient

        elif method == 'Momentum':
            error = X_train_b @ weights - y_train_scaled
            gradient = 2 * X_train_b.T @ error / n_samples
            velocity = 0.9 * velocity + gradient
            weights -= current_eta * velocity

        elif method == 'Adam':
            error = X_train_b @ weights - y_train_scaled
            gradient = 2 * X_train_b.T @ error / n_samples
            first_moment = 0.9 * first_moment + 0.1 * gradient
            second_moment = 0.999 * second_moment + 0.001 * gradient ** 2
            first_moment_corrected = first_moment / (1 - 0.9 ** iteration)
            second_moment_corrected = second_moment / (1 - 0.999 ** iteration)
            weights -= current_eta * first_moment_corrected / (
                np.sqrt(second_moment_corrected) + 1e-8
            )

        train_loss_scaled = np.mean(
            (X_train_b @ weights - y_train_scaled) ** 2
        )

        if not np.isfinite(train_loss_scaled):
            return weights, iteration, False

        loss_change = abs(previous_loss - train_loss_scaled)
        if (
            np.isfinite(previous_loss)
            and loss_change <= tol * max(1.0, previous_loss)
        ):
            stable_iterations += 1
        else:
            stable_iterations = 0

        if stable_iterations >= patience:
            return weights, iteration, True

        previous_loss = train_loss_scaled

    return weights, max_iter, True

eta_grid = np.logspace(-5, 0, 11)
lambda_grid = np.logspace(-5, 0, 11)
initial_eta = 0.1
experiment_results = {}

def run_experiment(method, variable_step=False):
    grid = lambda_grid if variable_step else eta_grid
    candidates = []

    for value in grid:
        eta = initial_eta if variable_step else value
        decay = value if variable_step else 0.0
        weights, iterations, valid = fit_linear_model(
            method, eta, decay
        )

        if valid:
            loss_train, r2_train = calculate_metrics(
                X_train_b, y_train, weights
            )
            loss_val, _ = calculate_metrics(X_val_b, y_val, weights)
        else:
            loss_train, r2_train, loss_val = np.inf, -np.inf, np.inf

        candidates.append({
            'value': value,
            'weights': weights,
            'iterations': iterations,
            'Loss_train': loss_train,
            'R2_train': r2_train,
            'Loss_val': loss_val
        })

    best = min(candidates, key=lambda result: result['Loss_val'])
    loss_test, r2_test = calculate_metrics(
        X_test_b, y_test, best['weights']
    )
    key = f'{method} TimeDecayLR' if variable_step else f'{method} constant'
    step = (
        f'n(t) = {initial_eta} / (1 + {best["value"]:.6g} * t)'
        if variable_step else f'n = {best["value"]:.6g}'
    )

    experiment_results[key] = {
        'Метод': key,
        'Лучший шаг': step,
        'Loss_train': best['Loss_train'],
        'Loss_test': loss_test,
        'R^2 train': best['R2_train'],
        'R^2 test': r2_test,
        'Итерации на test': best['iterations']
    }

    summary = pd.DataFrame([{
        'Метод': key,
        'Лучший параметр': best['value'],
        'Функция шага': step,
        'Loss_train': best['Loss_train'],
        'R^2_train': best['R2_train'],
        'Loss_val': best['Loss_val'],
        'Loss_test': loss_test,
        'R^2_test': r2_test,
        'Итерации на test': best['iterations']
    }])
    display(summary.round(6))


Train: (400, 14) (400,)
Validation: (50, 14) (50,)
Test: (50, 14) (50,)


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [5]:
run_experiment('VGD')


,Метод,Лучший параметр,Функция шага,Loss_train,R^2_train,Loss_val,Loss_test,R^2_test,Итерации на test
0,VGD constant,0.003162,n = 0.00316228,19.932684,0.972246,15.297029,20.507176,0.966461,1000


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [6]:
run_experiment('VGD', variable_step=True)


,Метод,Лучший параметр,Функция шага,Loss_train,R^2_train,Loss_val,Loss_test,R^2_test,Итерации на test
0,VGD TimeDecayLR,0.1,n(t) = 0.1 / (1 + 0.1 * t),19.925306,0.972257,15.348726,20.717963,0.966116,405


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [7]:
run_experiment('SGD')


,Метод,Лучший параметр,Функция шага,Loss_train,R^2_train,Loss_val,Loss_test,R^2_test,Итерации на test
0,SGD constant,0.001,n = 0.001,19.941049,0.972235,15.130447,20.775109,0.966023,1000


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [8]:
run_experiment('SGD', variable_step=True)


,Метод,Лучший параметр,Функция шага,Loss_train,R^2_train,Loss_val,Loss_test,R^2_test,Итерации на test
0,SGD TimeDecayLR,0.316228,n(t) = 0.1 / (1 + 0.316228 * t),19.92552,0.972256,15.319228,20.740611,0.966079,1000


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [9]:
run_experiment('SAG')


,Метод,Лучший параметр,Функция шага,Loss_train,R^2_train,Loss_val,Loss_test,R^2_test,Итерации на test
0,SAG constant,0.00001,n = 1e-05,19.924997,0.972257,15.354375,20.737819,0.966084,1000


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [10]:
run_experiment('SAG', variable_step=True)


,Метод,Лучший параметр,Функция шага,Loss_train,R^2_train,Loss_val,Loss_test,R^2_test,Итерации на test
0,SAG TimeDecayLR,0.316228,n(t) = 0.1 / (1 + 0.316228 * t),19.924547,0.972258,15.374565,20.80155,0.965979,215


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [11]:
run_experiment('Momentum')


,Метод,Лучший параметр,Функция шага,Loss_train,R^2_train,Loss_val,Loss_test,R^2_test,Итерации на test
0,Momentum constant,0.000316,n = 0.000316228,19.929682,0.972251,15.31149,20.573817,0.966352,1000


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [12]:
run_experiment('Momentum', variable_step=True)


,Метод,Лучший параметр,Функция шага,Loss_train,R^2_train,Loss_val,Loss_test,R^2_test,Итерации на test
0,Momentum TimeDecayLR,0.031623,n(t) = 0.1 / (1 + 0.0316228 * t),19.924518,0.972258,15.371779,20.803217,0.965977,158


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [13]:
run_experiment('Adam')


,Метод,Лучший параметр,Функция шага,Loss_train,R^2_train,Loss_val,Loss_test,R^2_test,Итерации на test
0,Adam constant,0.003162,n = 0.00316228,19.924726,0.972257,15.363258,20.745497,0.966071,967


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [14]:
run_experiment('Adam', variable_step=True)


,Метод,Лучший параметр,Функция шага,Loss_train,R^2_train,Loss_val,Loss_test,R^2_test,Итерации на test
0,Adam TimeDecayLR,0.316228,n(t) = 0.1 / (1 + 0.316228 * t),19.925171,0.972257,15.354295,20.698408,0.966148,754


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [15]:
comparison = pd.DataFrame(experiment_results.values())
comparison = comparison.sort_values('Loss_test').reset_index(drop=True)
display(comparison.round(6))


,Метод,Лучший шаг,Loss_train,Loss_test,R^2 train,R^2 test,Итерации на test
0,VGD constant,n = 0.00316228,19.932684,20.507176,0.972246,0.966461,1000
1,Momentum constant,n = 0.000316228,19.929682,20.573817,0.972251,0.966352,1000
2,Adam TimeDecayLR,n(t) = 0.1 / (1 + 0.316228 * t),19.925171,20.698408,0.972257,0.966148,754
3,VGD TimeDecayLR,n(t) = 0.1 / (1 + 0.1 * t),19.925306,20.717963,0.972257,0.966116,405
4,SAG constant,n = 1e-05,19.924997,20.737819,0.972257,0.966084,1000
5,SGD TimeDecayLR,n(t) = 0.1 / (1 + 0.316228 * t),19.925520,20.740611,0.972256,0.966079,1000
6,Adam constant,n = 0.00316228,19.924726,20.745497,0.972257,0.966071,967
7,SGD constant,n = 0.001,19.941049,20.775109,0.972235,0.966023,1000
8,SAG TimeDecayLR,n(t) = 0.1 / (1 + 0.316228 * t),19.924547,20.801550,0.972258,0.965979,215
9,Momentum TimeDecayLR,n(t) = 0.1 / (1 + 0.0316228 * t),19.924518,20.803217,0.972258,0.965977,158


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

лучше всех получился VGD с шагом n = 0,00316228, у него Loss_test = 20,51 и R^2_test = 0,9665. R^2_train и R^2_test близки, поэтому переобучения не видно  
R^2_train показывает какую долю разброса ответов модель объфясняет на данных, на которых училась, R^2_test показывает то же самое на новых данных  
по R^2_train видно, насколько хорошо метод выучил train a по R^2_test насколько хорошо он обобщает. потому что высокий R^2_train при сильно меньшем R^2_test может означать переобучение